# Chapter 8 — RAG Generation
## Topic 8: RAG vs. Fine-Tuning — When Each Is the Right Answer

**Run cells in order.**

- Module 1: Setup — the decision-framework function from the theory
- Module 2: Apply the framework to this project's actual situation
- Module 3: Cost model — RAG's ongoing cost vs. fine-tuning's periodic cost
- Module 4: Gap-diagnosis simulation — routing a "remaining gap" to the right fix

**Install:** none beyond the standard library.


## Module 1: Setup — The Decision Framework

**Requires:** nothing

In [ ]:
"""
MODULE 1: Setup -- Decision Framework

Implements the theory's structured reasoning function: given a set of
YES/NO facts about the situation, produce a recommendation with explicit
reasons, rather than a black-box verdict.
"""

def should_consider_finetuning(
    knowledge_changes_frequently: bool,
    citation_and_verification_required: bool,
    measured_rag_gap_after_full_pipeline: float,
    gap_is_format_or_style_related: bool,
    gap_is_reasoning_pattern_related: bool,
) -> dict:
    reasons_against_finetuning = []
    reasons_for_finetuning = []

    if knowledge_changes_frequently:
        reasons_against_finetuning.append(
            "Frequent knowledge changes favor RAG\'s re-ingest-without-retrain model"
        )
    if citation_and_verification_required:
        reasons_against_finetuning.append(
            "Citation/faithfulness verification (Ch8 Topics 2, 4) requires RAG\'s "
            "context-injection architecture; fine-tuned weights have no equivalent "
            "checkable source"
        )
    if gap_is_format_or_style_related or gap_is_reasoning_pattern_related:
        reasons_for_finetuning.append(
            "Format/style/reasoning-pattern gaps are fine-tuning\'s actual strength"
        )
    if measured_rag_gap_after_full_pipeline < 0.05:
        reasons_against_finetuning.append(
            "Remaining gap after full RAG pipeline is small -- fine-tuning cost "
            "may not be justified"
        )
    elif measured_rag_gap_after_full_pipeline >= 0.05 and not (gap_is_format_or_style_related or gap_is_reasoning_pattern_related):
        reasons_against_finetuning.append(
            "Remaining gap is likely fact-related, not format/reasoning-related -- "
            "fine-tuning does not directly address knowledge-access gaps the way RAG does"
        )

    recommendation = (
        "Fine-tuning likely NOT justified as primary strategy"
        if len(reasons_against_finetuning) > len(reasons_for_finetuning)
        else "Fine-tuning worth piloting for the SPECIFIC identified gap"
    )

    return {
        "reasons_against_finetuning": reasons_against_finetuning,
        "reasons_for_finetuning": reasons_for_finetuning,
        "recommendation": recommendation,
    }


print("Decision framework loaded. Module 1 complete. Run Module 2.")


## Module 2: Apply the Framework to This Project's Actual Situation

**Requires:** Module 1

In [ ]:
"""
MODULE 2: Apply to This Project

Runs the framework against this project's REAL, established facts:
  - FD policy content changes over time (new products, rate revisions)
  - Citation/verification is close to a compliance requirement (regulated NBFC)
  - Chapter 2's measured gap (0.50-0.80 vs 0.97 MuRIL baseline) was a
    KNOWLEDGE-ACCESS gap, pre-RAG -- format/reasoning-related: NO
"""

this_project_analysis = should_consider_finetuning(
    knowledge_changes_frequently=True,
    citation_and_verification_required=True,
    measured_rag_gap_after_full_pipeline=0.03,  # illustrative post-RAG remaining gap
    gap_is_format_or_style_related=False,
    gap_is_reasoning_pattern_related=False,
)

print("Applying the framework to THIS project\'s actual, established facts:\n")
print("Reasons AGAINST fine-tuning as primary strategy:")
for r in this_project_analysis["reasons_against_finetuning"]:
    print(f"  - {r}")

print("\nReasons FOR fine-tuning (as a targeted addition):")
for r in this_project_analysis["reasons_for_finetuning"] or ["(none identified from current facts)"]:
    print(f"  - {r}")

print(f"\nRECOMMENDATION: {this_project_analysis['recommendation']}")

print("\n" + "-" * 65)
print("Contrast: a hypothetical DIFFERENT project where fine-tuning fits better")
print("-" * 65)

different_project = should_consider_finetuning(
    knowledge_changes_frequently=False,       # static domain knowledge
    citation_and_verification_required=False, # no compliance/audit requirement
    measured_rag_gap_after_full_pipeline=0.15,
    gap_is_format_or_style_related=True,       # e.g. needs a very specific output format
    gap_is_reasoning_pattern_related=False,
)
print(f"RECOMMENDATION: {different_project['recommendation']}")
print("(Static knowledge + no verification requirement + format-related gap")
print(" is exactly the profile where fine-tuning becomes the stronger choice --")
print(" this project\'s FD/NBFC domain profile is the OPPOSITE on every dimension.)")

print("\nModule 2 complete. Run Module 3.")


## Module 3: Cost Model — RAG's Ongoing Cost vs. Fine-Tuning's Periodic Cost

**Requires:** nothing standalone

In [ ]:
"""
MODULE 3: Cost Model

Illustrative cost comparison over a 12-month horizon: RAG's per-query
inference cost (compounding with volume) vs. fine-tuning's periodic
retraining cost (compounding with how often the knowledge changes).
"""

def project_costs(months: int, daily_queries: int,
                   rag_extra_tokens_per_query: int, cost_per_million_tokens: float,
                   finetune_cost_per_run: float, retrain_frequency_months: float) -> dict:

    total_queries = daily_queries * 30 * months
    rag_total_extra_tokens = total_queries * rag_extra_tokens_per_query
    rag_total_cost = (rag_total_tokens := rag_total_extra_tokens) / 1_000_000 * cost_per_million_tokens

    n_retrains = months / retrain_frequency_months
    finetune_total_cost = n_retrains * finetune_cost_per_run

    return {
        "months": months,
        "total_queries": total_queries,
        "rag_total_cost_usd": rag_total_cost,
        "finetune_retrains": n_retrains,
        "finetune_total_cost_usd": finetune_total_cost,
    }


# Illustrative parameters -- NOT precise, purely to show the SHAPE of the trade-off
result_dynamic = project_costs(
    months=12, daily_queries=10_000,
    rag_extra_tokens_per_query=2000,      # Topic 1's typical context block size
    cost_per_million_tokens=1.0,          # claude-haiku-4-5-20251001 input pricing
    finetune_cost_per_run=500,            # illustrative training run cost
    retrain_frequency_months=2,           # DYNAMIC content -- retrain every 2 months
)

result_static = dict(result_dynamic)
result_static["finetune_retrains"] = 12 / 12  # STATIC content -- retrain once a year
result_static["finetune_total_cost_usd"] = result_static["finetune_retrains"] * 500

print("Scenario A: DYNAMIC knowledge (this project\'s actual FD policy situation)")
print(f"  RAG total cost over 12 months:        ${result_dynamic['rag_total_cost_usd']:,.2f}")
print(f"  Fine-tuning total cost over 12 months: ${result_dynamic['finetune_total_cost_usd']:,.2f} "
      f"({result_dynamic['finetune_retrains']:.0f} retrains)")

print("\nScenario B: STATIC knowledge (hypothetical, for contrast)")
print(f"  RAG total cost over 12 months:        ${result_static['rag_total_cost_usd']:,.2f} (unchanged)")
print(f"  Fine-tuning total cost over 12 months: ${result_static['finetune_total_cost_usd']:,.2f} "
      f"({result_static['finetune_retrains']:.0f} retrain)")

print("""
Interpretation: RAG's cost is roughly CONSTANT regardless of how often
underlying knowledge changes (Scenario A and B show the same RAG cost).
Fine-tuning's cost SCALES with how often knowledge changes -- Scenario A
(this project's actual dynamic-policy situation) pays for 6 retrains,
Scenario B (static) pays for only 1. This is the concrete cost mechanism
behind the theory's "dynamic knowledge favors RAG" argument -- it's not
just an abstract preference, it's a directly quantifiable cost difference.

NOTE: these numbers are illustrative, not precise -- the actual costs
depend on real token volumes, real training infrastructure costs (Chapter
17-19), and real retraining frequency, which should be measured, not
assumed, before this comparison drives a real budget decision.
""")

print("Module 3 complete. Run Module 4.")


## Module 4: Gap-Diagnosis Simulation — Routing to the Right Fix

**Requires:** Module 1

In [ ]:
"""
MODULE 4: Gap-Diagnosis Simulation

Simulates the Section 8 "Advanced" interview answer as executable logic:
given a diagnosed gap TYPE, route to the correct remediation -- NOT every
gap should trigger the same response.
"""

def route_gap_to_fix(gap_type: str) -> str:
    routing = {
        "retrieval_quality": "Fix belongs in Chapter 7 (BM25/dense/hybrid/reranking) -- "
                              "NOT a Chapter 8 generation problem, NOT a fine-tuning problem.",
        "faithfulness_inconsistency": "Fine-tuning IN SERVICE OF RAG -- train on examples of "
                                       "correctly-cited, well-grounded answers to improve "
                                       "consistency, while KEEPING RAG\'s retrieval mechanism.",
        "knowledge_staleness": "Knowledge base governance (Chapter 4/6 ingestion, freshness "
                                "tracking) -- NEITHER better retrieval NOR fine-tuning fixes "
                                "genuinely outdated source content.",
        "output_format_inconsistency": "Fine-tuning is well-suited here -- teaching consistent "
                                        "structured output is fine-tuning\'s genuine strength.",
    }
    return routing.get(gap_type, "Unrecognized gap type -- diagnose further before acting.")


test_gaps = [
    "retrieval_quality",
    "faithfulness_inconsistency",
    "knowledge_staleness",
    "output_format_inconsistency",
]

print(f"{'Diagnosed gap type':<28} | Correct remediation")
print("-" * 90)
for gap in test_gaps:
    fix = route_gap_to_fix(gap)
    print(f"{gap:<28} | {fix}")
    print()

print("=" * 70)
print("CHAPTER 8 TOPIC 8 SUMMARY")
print("=" * 70)
print("""
RAG and fine-tuning solve DIFFERENT problems -- not strictly competing.
This project\'s profile (dynamic policy content, citation/compliance
  requirement, pre-RAG gap was knowledge-access not format/reasoning)
  strongly favors RAG as PRIMARY architecture -- confirmed by the
  framework in Modules 1-2.
Cost model (Module 3): RAG cost stays roughly constant regardless of
  knowledge-change frequency; fine-tuning cost scales directly with it --
  a concrete, quantifiable version of the theory\'s qualitative argument.
Gap routing (Module 4): NOT every remaining problem should trigger
  fine-tuning -- retrieval gaps belong in Chapter 7, staleness gaps belong
  in knowledge base governance, and only format/consistency gaps are
  fine-tuning\'s genuine strength.
Most defensible fine-tuning use case for this project: improving citation/
  faithfulness CONSISTENCY on top of RAG\'s factual grounding -- fine-tuning
  IN SERVICE OF RAG, not a replacement for RAG\'s knowledge-access role.

This closes Chapter 8. Chapter 9 (Retrieval -- Putting It to Work) wires
this generation layer into the full agent architecture from Chapter 3.
""")
